# PART 3.2: Word2Vec

In [1]:
import os, sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk

nltk.download("punkt")
nltk.download("punkt_tab")

sys.path.append(os.path.abspath(os.path.join("..", "..")))

from collections import defaultdict, Counter
from dotenv import load_dotenv

from myapp.search import load_corpus as lc
from project_progress.part_1.data_prep import (
    corpus_df_loading,
    build_terms,
    join_build_terms,
)
from project_progress.part_2.index_tf_idf import create_index_tf_idf

from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import preprocess_string


load_dotenv()  # take environment variables from .env

[nltk_data] Downloading package punkt to /home/nara-
[nltk_data]     dellans/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/nara-
[nltk_data]     dellans/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/nara-
[nltk_data]     dellans/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

Functions to store the time-consuming processing in files to load faster after:

In [2]:
products_filepath = "../../data/products.json"
products_numeric_data_filepath = "../../data/products_numeric_data.json"


def dump_data(data, filepath):
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def load_data(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data


def get_or_create(filepath, compute_data_function):
    if os.path.exists(filepath):
        with open(filepath, "r", encoding="utf-8") as f:
            return json.load(f)
    else:
        data = compute_data_function()
        with open(filepath, "w+", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
        return data

Now, we load the corpus:

In [3]:
def process_products(corpus, preprocess=preprocess_string):
    """
    Function that loads the products of the corpus as a dictionary of pid -> list (of words in categorical data as lists of processed tokens per affine categories)

    :param corpus: Corpus with all the products and all their data.
    :return product_sentences: (dict) pid -> lists of tokens (all preprocessed terms of the categorical data per grouped subcategories)
    """
    products_sentences = {}
    for product in list(corpus.values()):

        # Process the documents categorical fields as in the creation of the inverted index, but concatenate all the terms in a single string (in this case we do not care about the fields)
        subgroups = [
            " ".join([product.title, product.description]),
            " ".join([product.brand, product.category, product.sub_category]),
            " ".join([product.seller, 
                      " ".join([detail for detail in product.product_details.values()])]
                      )
            ]
        products_sentences[product.pid] = [preprocess(group) for group in subgroups]

    return products_sentences

In [4]:
json_path = "../../data/fashion_products_dataset.json"
corpus = corpus_df_loading(json_path)

# Preprocess the corpus to get the products
product_sentences = get_or_create(products_filepath, lambda: process_products(corpus))

Then, we compute the inverted index:

In [5]:
index, index2title, tf, df, idf = create_index_tf_idf(corpus=corpus)

## Document Representation using Word2Vec 

First, we will build the list of sentences for the model and train the model with them. 

In [6]:
def get_training_sentences(product_sentences):
    """
    Function that builds the training sentences from the product sentences.

    :param product_sentences: Dictionary of pid -> list of tokenized sentences.
    :return sentences: List of tokenized sentences for training the Word2Vec model.
    """
    sentences = []

    for prod_sentences in product_sentences.values():
        sentences.extend(prod_sentences)

    return sentences

In [7]:
def train_word2vec_model(sentences, vector_size=100, window=7, min_count=5, negative=10, sg=1):
    """
    Function that trains a Word2Vec model on the given sentences.

    :param sentences: List of tokenized sentences for training the Word2Vec model.
    :param vector_size: Dimensionality of the word vectors.
    :param window: Maximum distance between the current and predicted word within a sentence.
    :param min_count: Ignores all words with total frequency lower than this.
    :param negative: If > 0, negative sampling will be used, the int for the number of negative samples.
    :param sg: Training algorithm: 1 for skip-gram; otherwise CBOW.
    :param epochs: Number of iterations (epochs) over the corpus.
    :return model: Trained Word2Vec model.
    """
    model = Word2Vec(
        sentences=sentences,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        negative=negative,
        sg=sg, 
    )

    return model


In [8]:
training_sentences = get_training_sentences(product_sentences)

word2vec_model = train_word2vec_model(training_sentences)

Then, we generate the embeddings for all the products 

In [9]:
def build_text_terms(sentences):
    """
    Function that builds a list of terms from the given sentences.
    
    :param sentences: List of tokenized sentences.
    :return term_list: List of terms.
    """
    term_list = []
    for sentence in sentences:
        term_list.extend(sentence)

    return term_list

In [10]:
def get_embedding(text, model:Word2Vec):
    """
    Function that computes the embedding of a document by averaging the embeddings of its terms.

    :param text: List of tokenized sentences representing the document.
    :param model: Trained Word2Vec model.
    :return embedding: Numpy array representing the document embedding.
    """
    doc_terms = build_text_terms(text)
    word_embeddings = [model.wv[word] if word in model.wv else np.zeros(model.vector_size) for word in doc_terms]

    embedding = np.mean(word_embeddings, axis=0)
    return embedding

def get_document_embeddings(word2vec_model, product_sentences):
    """
    Function that computes the document embeddings for all products.
    
    :param word2vec_model: Trained Word2Vec model.
    :param product_sentences: Dictionary of pid -> list of tokenized sentences.
    :return doc_embeddings: Dictionary of pid -> document embedding.
    """
    doc_embeddings = {}
    for pid, sentences in product_sentences.items():
        doc_embeddings[pid] = get_embedding(sentences, word2vec_model)
    return doc_embeddings


In [11]:
doc_embeddings = get_document_embeddings(word2vec_model, product_sentences)

### Word2Vec + cosine similarity

We are going to use the same queries from [PART 2](../part_2/indexing_and_evaluation.ipynb) of the project, defined in the _Query Selection_ section, to test how the word embedding works.

In [12]:
def cosine_similarity(document_representation, query_representation):
    """
    Function that computes the cosine similarity between a document and a query representation.

    :param document_representation: Numpy array representing the document.
    :param query_representation: Numpy array representing the query.
    :return similarity: Cosine similarity score.
    """
    dot_product = np.dot(document_representation, query_representation)
    norm_document = np.linalg.norm(document_representation)

    if norm_document == 0:
        return 0.0
    
    return dot_product / (norm_document)

In [13]:
def rank_documents(w2v_model, doc2vec, query, preprocess=preprocess_string):
    """
    Function that ranks documents based on their cosine similarity to the query.
    :param w2v_model: Trained Word2Vec model.
    :param doc2vec: Dictionary of pid -> document embedding.
    :param query: Query string.
    :param preprocess: Preprocessing function to apply to the query.
    :return sim_scores: List of [similarity score, pid] sorted in descending order
    """
    query_terms = preprocess_string(query)
    query_embedding = get_embedding(query_terms, w2v_model)

    sim_scores = []
    for pid, doc_embedding in doc2vec.items():
        score = cosine_similarity(doc_embedding,query_embedding)
        sim_scores.append([score, pid])
    
    sim_scores.sort(key = lambda x: x[0], reverse=True)

    return sim_scores

In [14]:
def print_ranking(scores, index2title):
    """
    Function to print the ranking for an arbitrary algorithm.

    :param scores: (list) of value pairs [score, pid] with score being the score for the document pid
    :param index2title: (dict) pid -> (string) content of the "title" field of the product
    """
    for idx, (score, pid) in enumerate(scores):
        print(f"{idx+1:4}. [score={score:.3f}] {index2title[pid]}")

In [15]:
def filter(query, products):
    """
    The output is the list of documents that contain ALL query terms.

    :param query: (string) query
    :param products: (Dict) pid -> document text
    :return selected_docs: (List) of documents' ids that contain all query terms
    """

    query_terms = build_terms(query)  # tokenize query
    selected_docs = []

    for pid, prod_terms in products.items():
        # check if ALL query terms are in this document
        if all(term in prod_terms for term in query_terms):
            selected_docs.append(pid)

    return query_terms, selected_docs

In [16]:
queries = [
    "western leather jacket men",  # context, material, specific cloth, gender
    "cotton innerwear man",  # material, specific cloth, gender
    "yellow black t-shirt women xl",  # adjectives, specific cloth, gender, size
    "casual comfortable blue trousers women",  # context, adjectives, specific cloth, gender
    "breathable sports clothes winter",
]  # adjectives, context, general clothes, context

In [17]:
for query in queries:
    print(f"\n\033[92mQUERY: {query}\033[0m")

    '''STEP 1: filter the products'''
    _, filtered_docs = filter(query=query, products=product_sentences)
    filtered_doc_embeddings = {pid: doc_embeddings[pid] for pid in filtered_docs}
    if (len(filtered_doc_embeddings) == 0):   # Compute ranking only if we found documents during filtering
        print("\n\033[91mNo results!\033[0m")
    else:
        '''STEP 2: rank the filtered products'''
        scores_w2vcossim = rank_documents(word2vec_model, filtered_doc_embeddings, query)
        print_ranking(scores=scores_w2vcossim, index2title=index2title)



QUERY: western leather jacket men
   1. [score=1.125] Full Sleeve Solid Men Leather Jacket
   2. [score=1.124] Full Sleeve Solid Men Leather Jacket
   3. [score=1.124] Full Sleeve Solid Men Leather Jacket
   4. [score=1.124] Full Sleeve Solid Men Leather Jacket
   5. [score=1.124] Full Sleeve Solid Men Leather Jacket
   6. [score=1.119] Solid Men Henley Neck Green T-Shirt  (Pack of 3)
   7. [score=1.118] Full Sleeve Solid Men Leather Jacket
   8. [score=1.117] Solid Men Henley Neck Red T-Shirt  (Pack of 3)
   9. [score=1.117] Full Sleeve Solid Men Leather Jacket
  10. [score=1.117] Full Sleeve Solid Men Leather Jacket
  11. [score=1.117] Full Sleeve Solid Men Leather Jacket
  12. [score=1.114] Full Sleeve Solid Men Riding Jacket
  13. [score=1.114] Full Sleeve Solid Men Riding Jacket
  14. [score=1.108] Full Sleeve Color Block, Applique Men Leather Jacket
  15. [score=1.107] Full Sleeve Color Block, Applique Men Leather Jacket
  16. [score=1.107] Full Sleeve Washed Men Denim Jacket
  